In [1]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse.linalg import eigsh  # ARPACK on sparse matrices

In [5]:
# --- Pretty, slide-ready degree plots + spectral radius for ER, WS, BA ---
# Requires: networkx, numpy, matplotlib, scipy
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse.linalg import eigsh  # ARPACK on sparse matrices

# -------------------------- helpers ---------------------------------
def degree_array(G):
    return np.fromiter((d for _, d in G.degree()), dtype=int)

def ccdf_from_degrees(k):
    k = np.asarray(k)
    vals = np.unique(k)
    ccdf = np.array([(k >= x).mean() for x in vals])  # P(K >= x)
    return vals, ccdf

def lambda_max(G, tol=1e-6):
    A = nx.to_scipy_sparse_array(G, dtype=float, format="csr")
    val = eigsh(A, k=1, which="LA", return_eigenvectors=False, tol=tol)[0]
    return float(val)

def pretty_axes(ax):
    ax.grid(True, alpha=0.25, linewidth=0.8)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="both", labelsize=11)

# ----------------------- graph builders ------------------------------
def build_graphs(N=5000, k_target=12, ws_rewire=0.1, seed=123):
    # ER: p for expected average degree ~ k_target
    p = k_target / (N - 1)
    G_er = nx.fast_gnp_random_graph(N, p, seed=seed)

    # WS: nearest even K to k_target
    K = int(round(k_target))
    if K % 2 == 1:
        K += 1
    K = max(2, min(K, N - 2))
    G_ws = nx.watts_strogatz_graph(N, K, ws_rewire, seed=seed + 1)

    # BA: average degree ~ 2m
    m_ba = max(1, int(round(k_target / 2)))
    G_ba = nx.barabasi_albert_graph(N, m_ba, seed=seed + 2)

    avg = lambda G: np.mean(degree_array(G))
    info = {
        "ER": {"G": G_er, "params": {"p": p}},
        "WS": {"G": G_ws, "params": {"K": K, "q": ws_rewire}},
        "BA": {"G": G_ba, "params": {"m": m_ba}},
    }
    for name, d in info.items():
        d["avg_deg"] = float(avg(d["G"]))
        d["lambda_max"] = lambda_max(d["G"])
    return info

# ------------------------- plotting ---------------------------------


# def save_pmf_plot(name, d, k_target, outdir="figs"):
    from collections import Counter
    from matplotlib.ticker import MultipleLocator
    Path(outdir).mkdir(parents=True, exist_ok=True)

    G = d["G"]
    k = degree_array(G)

    # Compute PMF
    counts = Counter(k)
    degrees = np.array(sorted(counts.keys()))
    pmf = np.array([counts[kk] / len(k) for kk in degrees])

    fig, ax = plt.subplots(figsize=(6.5, 4.5))

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    fig.subplots_adjust(left=0.18)

    # markers only (discrete PMF)
    ax.plot(degrees, pmf, marker="o", ms=4, lw=0)

    # Labels
    ax.set_xlabel("Degree", fontsize=16)
    ax.set_ylabel(r"$P(K = k)$", fontsize=16)

    # ---- FIXED x-axis range and integer ticks ----
    ax.set_xlim(0, 18)
    ax.xaxis.set_major_locator(MultipleLocator(1))

    # Optional: remove minor ticks on x (keeps it clean)
    ax.xaxis.set_minor_locator(MultipleLocator(1))

    # Vertical dashed line for target degree
    ax.axvline(k_target, ls="--", lw=1, color="black", alpha=0.6)

    # Styling
    ax.tick_params(axis='both', which='major', labelsize=16)
    pretty_axes(ax)

    fname = Path(outdir) / f"pmf_{name}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=300, transparent=True)
    plt.close(fig)

    print(f"Saved {fname}")
    return str(fname)

def save_pmf_plot(name, d, k_target, outdir="figs"):
    from collections import Counter
    from matplotlib.ticker import MultipleLocator, FormatStrFormatter
    Path(outdir).mkdir(parents=True, exist_ok=True)

    G = d["G"]
    k = degree_array(G)

    # Compute PMF
    counts = Counter(k)
    degrees = np.array(sorted(counts.keys()))
    pmf = np.array([counts[kk] / len(k) for kk in degrees])

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    fig.subplots_adjust(left=0.18)  # fixed margin for paper alignment

    # Discrete PMF: markers only
    ax.plot(degrees, pmf, marker="o", ms=4, lw=0)

    # Labels
    ax.set_xlabel("Degree", fontsize=16)
    ax.set_ylabel(r"$P(K = k)$", fontsize=16)

    # Fixed x-axis range and integer ticks
    ax.set_xlim(0, 18)
    ax.xaxis.set_major_locator(MultipleLocator(1))

    # ---- FIXED y-axis formatting (KEY CHANGE) ----
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

    # Vertical dashed line for target degree
    ax.axvline(k_target, ls="--", lw=1, color="black", alpha=0.6)

    # Styling
    ax.tick_params(axis='both', which='major', labelsize=16)
    pretty_axes(ax)

    fname = Path(outdir) / f"pmf_{name}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=300, transparent=True)
    plt.close(fig)

    print(f"Saved {fname}")
    return str(fname)


def save_pmf_loglog_plot(name, d, k_target, outdir="figs"):
    from collections import Counter
    from matplotlib.ticker import LogFormatterMathtext
    Path(outdir).mkdir(parents=True, exist_ok=True)

    G = d["G"]
    k = degree_array(G)

    # PMF
    counts = Counter(k)
    degrees = np.array(sorted(counts.keys()))
    pmf = np.array([counts[kk] / len(k) for kk in degrees])

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    fig.subplots_adjust(left=0.18)

    # Discrete PMF: markers only
    ax.plot(degrees, pmf, marker="o", ms=4, lw=0)

    # Log–log scale
    ax.set_xscale("log")
    ax.set_yscale("log")

    # Labels
    ax.set_xlabel("Degree", fontsize=16)
    ax.set_ylabel(r"$P(K = k)$", fontsize=16)

    # Consistent log tick formatting
    ax.yaxis.set_major_formatter(LogFormatterMathtext())

    # ---- Target degree marker (NOW INCLUDED) ----
    ax.axvline(k_target, ls="--", lw=1.0, color="black", alpha=0.6, zorder=0)

    # Optional annotation (cleaner than legend)
    # ax.text(
    #     k_target,
    #     ax.get_ylim()[0] * 1.5,
    #     r"$k_{\mathrm{target}}$",
    #     ha="center",
    #     va="bottom",
    #     fontsize=14
    # )

    # Styling
    ax.tick_params(axis='both', which='major', labelsize=16)
    pretty_axes(ax)

    fname = Path(outdir) / f"pmf_loglog_{name}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=300, transparent=True)
    plt.close(fig)

    print(f"Saved {fname}")
    return str(fname)




# -------------------------- save adjlists ----------------------------
def save_adjlists(graphs, outdir="graphs"):
    Path(outdir).mkdir(parents=True, exist_ok=True)
    for name, d in graphs.items():
        path = Path(outdir) / f"{name}-{len(d['G'])}.adjlist"
        nx.write_adjlist(d["G"], path)
        print(f"Saved adjacency list: {path}")

# -------------------------- run example ------------------------------
if __name__ == "__main__":
    N = 10000
    k_target = 4
    ws_rewire = 0.1
    seed = 123

    graphs = build_graphs(N=N, k_target=k_target, ws_rewire=ws_rewire, seed=seed)

    print(f"Target average degree: {k_target}")
    for name, d in graphs.items():
        print(f"{name}: avg_deg={d['avg_deg']:.3f}, λ_max={d['lambda_max']:.3f}, params={d['params']}")

    # Separate, slide-ready images
    for name, d in graphs.items():
        save_pmf_plot(name, d, k_target=k_target, outdir="figs")
        # save_ccdf_plot(name, d, outdir="figs")
        save_pmf_loglog_plot(name, d, k_target=k_target, outdir="figs")

    # Save adjlists for later use
    # save_adjlists(graphs, outdir="graphs")  

Target average degree: 4
ER: avg_deg=3.968, λ_max=5.235, params={'p': 0.00040004000400040005}
WS: avg_deg=4.000, λ_max=4.195, params={'K': 4, 'q': 0.1}
BA: avg_deg=3.999, λ_max=15.891, params={'m': 2}
Saved figs\pmf_ER.png
Saved figs\pmf_loglog_ER.png
Saved figs\pmf_WS.png
Saved figs\pmf_loglog_WS.png
Saved figs\pmf_BA.png
Saved figs\pmf_loglog_BA.png


# ER

In [10]:
G = nx.read_adjlist("graphs/ER-1000.adjlist", nodetype=int)
A = nx.to_numpy_array(G, dtype=int)

eigenvalues, _ = np.linalg.eig(A)
lambda_max_A = np.max(eigenvalues)
tau_c = 1 / lambda_max_A
print('tau_c =', tau_c)

m = 4
a = 0.1
tau_ct = 1 / (lambda_max_A + 2*m*a)
print('tau_ct =', tau_ct, 'with m =', m)

beta = 0.14
mu = 1
tau = beta/mu
print('tau =', tau)

m_c = ((1 / tau) - (lambda_max_A)) / (2*a)
print('m_c =', m_c)

print('lamba_max =', lambda_max_A)

tau_c = (0.18809970822715075+0j)
tau_ct = (0.16349675473739825+0j) with m = 4
tau = 0.14
m_c = (9.132638952973572+0j)
lamba_max = (5.316329352262428+0j)


# WS

In [13]:
G = nx.read_adjlist("graphs/WS-1000.adjlist", nodetype=int)
A = nx.to_numpy_array(G, dtype=int)

eigenvalues, _ = np.linalg.eig(A)
lambda_max_A = np.max(eigenvalues)
tau_c = 1 / lambda_max_A
print('tau_c =', tau_c)

m = 5
a = 0.1
tau_ct = 1 / (lambda_max_A + 2*m*a)
print('tau_ct =', tau_ct, 'with m =', m)

beta = 0.14
mu = 1
tau = beta/mu
print('tau =', tau)

m_c = ((1 / tau) - (lambda_max_A)) / (2*a)
print('m_c =', m_c)

print('lamba_max =', lambda_max_A)

tau_c = 0.2404646330924437
tau_ct = 0.19385045464212233 with m = 5
tau = 0.14
m_c = 14.921207182534179
lamba_max = 4.158615706350306


# BA

In [12]:
G = nx.read_adjlist("graphs/BA-1000.adjlist", nodetype=int)
A = nx.to_numpy_array(G, dtype=int)

eigenvalues, _ = np.linalg.eig(A)
lambda_max_A = np.max(eigenvalues)
tau_c = 1 / lambda_max_A
print('tau_c =', tau_c)

m = 5
a = 0.1
tau_ct = 1 / (lambda_max_A + 2*m*a)
print('tau_ct =', tau_ct, 'with m =', m)

beta = 0.09
mu = 1
tau = beta/mu
print('tau =', tau)

m_c = ((1 / tau) - (lambda_max_A)) / (2*a)
print('m_c =', m_c)

print('lamba_max =', lambda_max_A)

tau_c = (0.10137122117817814+0j)
tau_ct = (0.0920409206531995+0j) with m = 5
tau = 0.09
m_c = (6.231892075053516+0j)
lamba_max = (9.864732696100408+0j)
